In [ ]:
import dask.dataframe as dd
from dask.distributed import Client, progress
import time
import numpy as np

# Setup Dask distributed client for parallel processing at the start
client = Client(memory_limit='8GB')
print(f"Dask dashboard available at: {client.dashboard_link}")
client

/home/artificialstupidity/ttc-bus-delay-prediction/.venv/lib/python3.12/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 35499 instead
  warnings.warn(
/home/artificialstupidity/ttc-bus-delay-prediction/.venv/lib/python3.12/site-packages/distributed/diagnostics/nvml.py:14: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml
/home/artificialstupidity/ttc-bus-delay-prediction/.venv/lib/python3.12/site-packages/distributed/diagnostics/nvml.py:14: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml
/home/artificialstupidity/ttc-bus-delay-pred

Dask dashboard available at: http://127.0.0.1:35499/status


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:35499/status,
Dashboard: http://127.0.0.1:35499/status,Workers: 4
Total threads: 16,Total memory: 29.80 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:42565,Workers: 0
Dashboard: http://127.0.0.1:35499/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:33537,Total threads: 4
Dashboard: http://127.0.0.1:36829/status,Memory: 7.45 GiB
Nanny: tcp://127.0.0.1:39291,


2026-02-24 21:53:47,129 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 2151077bb3d9105a6919cb7ffd80b77d initialized by task ('shuffle-transfer-2151077bb3d9105a6919cb7ffd80b77d', 69) executed on worker tcp://127.0.0.1:44991
2026-02-24 21:54:15,980 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 2151077bb3d9105a6919cb7ffd80b77d deactivated due to stimulus 'task-finished-1771949355.9766972'
2026-02-24 21:54:33,605 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 7a1f2f59fb2181eed4b1eefe82fc924d initialized by task ('shuffle-transfer-7a1f2f59fb2181eed4b1eefe82fc924d', 74) executed on worker tcp://127.0.0.1:33537
2026-02-24 21:55:21,027 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 7a1f2f59fb2181eed4b1eefe82fc924d deactivated due to stimulus 'task-finished-1771949421.0254915'
2026-02-24 21:55:27,062 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle bb6db21ba174cd1c3978ce9ffae62ce5 initialized by task ('shuffle-transfer-bb6db21ba174

In [65]:
df = dd.read_csv("./tmp/unfv.csv", 
                 usecols=[
                     "trip_id",
                     "time",
                     "?column?",
                     'stopid',
                     'route',
                     'thrusteet'
                 ],
                 dtype={
                     "trip_id": "object",
                     'stopid': 'object',
                    }
                 )

# df = dd.read_csv("./tmp/unfv.csv",
#                  dtype={'direction': 'float64',
#        'timespent': 'float64',
#        'trip_id': 'object',
#        'stopid': 'object',})

In [66]:
df.head()

,route,trip_id,stopid,thrusteet,time,?column?
0,86,88277020,3803_1,Kennedy Station,2025-01-14 17:39:53,2025-01-14 17:41:00
1,86,88277020,3803_1,Kennedy Station,2025-01-14 17:39:53,2025-01-14 17:41:00
2,86,88277020,3803_1,Kennedy Station,2025-01-14 17:39:53,2025-01-14 17:41:00
3,86,88277020,3803_1,Kennedy Station,2025-01-14 17:39:53,2025-01-14 17:41:00
4,86,88277020,3803_1,Kennedy Station,2025-01-14 17:39:53,2025-01-14 17:41:00


In [67]:
df = df.rename(columns={
    "time": "actual_arrival",
    "?column?": "planned_arrival"
})

In [68]:
df.thrusteet.value_counts().compute()

thrusteet
Gordon Baker Rd                               82307
Harold Evans Cres                             13704
Highland Creek Overpass                      535734
Toryork Dr                                    73204
Fenmar Dr                                     14594
Eglinton Ave East                           5380318
Nova Scotia Ave                               14875
Finch Station                               4591108
Humberline Dr                               1019458
Princes' Blvd                                 27209
Baldoon Rd                                    27626
Humberwood Blvd Loop                         579095
Finch Ave West - St Wilfreds School          487319
Coronation Dr                                152351
Finch Ave West - Finch/Weston Centre         211794
Meadowvale Rd                               1241537
Beechgrove Dr                                 50582
Finch Ave East - St Aidan                    210500
Park Rd (Zoo)                                 82108
La

We will be using route 29 and 29

In [69]:
df = df[df['route'].isin([29, 39])]

In [70]:
# df.shape[0].compute()

In [71]:
# change to python datetime
df['actual_arrival'] = dd.to_datetime(df['actual_arrival'], format='mixed')
df['planned_arrival'] = dd.to_datetime(df['planned_arrival'], format='mixed')

In [72]:
# df.head(5)

In [73]:
df = df.drop_duplicates(subset=['stopid', 'trip_id', 'actual_arrival', 'planned_arrival', 'route'])
df = df.sort_values('actual_arrival')

In [74]:
# df.head(5)

In [75]:
df['hour'] = df['planned_arrival'].dt.hour
df['day_of_week'] = df['planned_arrival'].dt.dayofweek
df['month'] = df['planned_arrival'].dt.month
df['day_of_year'] = df['planned_arrival'].dt.dayofyear

df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
df['delay_seconds'] = (df['actual_arrival'] - df['planned_arrival']).dt.total_seconds()

In [76]:
df.dtypes

route                      float64
trip_id            string[pyarrow]
stopid             string[pyarrow]
thrusteet          string[pyarrow]
actual_arrival      datetime64[ns]
planned_arrival     datetime64[ns]
hour                         int32
day_of_week                  int32
month                        int32
day_of_year                  int32
hour_sin                   float64
hour_cos                   float64
day_of_week_sin            float64
day_of_week_cos            float64
delay_seconds              float64
dtype: object

In [77]:
df.head()

,route,trip_id,stopid,thrusteet,actual_arrival,planned_arrival,hour,day_of_week,month,day_of_year,hour_sin,hour_cos,day_of_week_sin,day_of_week_cos,delay_seconds
78721,29.0,172354080,3880_0,Wilson Station,2025-01-01 00:01:20.000,2025-01-01 00:02:00,0,2,1,1,0.0,1.0,0.974928,-0.222521,-40.000
78822,29.0,52553080,2728_1,Princes' Gates Loop,2025-01-01 00:01:34.029,2025-01-01 00:03:00,0,2,1,1,0.0,1.0,0.974928,-0.222521,-85.971
160789,39.0,76911080,3784_1,Finch Station,2025-01-01 00:02:41.000,2025-01-01 00:05:00,0,2,1,1,0.0,1.0,0.974928,-0.222521,-139.000
78826,29.0,52553080,3993,Princes' Blvd,2025-01-01 00:03:46.145,2025-01-01 00:03:52,0,2,1,1,0.0,1.0,0.974928,-0.222521,-5.855
78827,29.0,52553080,34800,Princes' Blvd,2025-01-01 00:06:13.000,2025-01-01 00:05:45,0,2,1,1,0.0,1.0,0.974928,-0.222521,28.000


In [ ]:
# df.head(5)

,route,trip_id,stopid,actual_arrival,planned_arrival,hour,day_of_week,month,day_of_year,hour_sin,hour_cos,day_of_week_sin,day_of_week_cos,delay_seconds
78721,29.0,172354080,3880_0,2025-01-01 00:01:20.000,2025-01-01 00:02:00,0,2,1,1,0.0,1.0,0.974928,-0.222521,-40.000
78822,29.0,52553080,2728_1,2025-01-01 00:01:34.029,2025-01-01 00:03:00,0,2,1,1,0.0,1.0,0.974928,-0.222521,-85.971
160789,39.0,76911080,3784_1,2025-01-01 00:02:41.000,2025-01-01 00:05:00,0,2,1,1,0.0,1.0,0.974928,-0.222521,-139.000
78826,29.0,52553080,3993,2025-01-01 00:03:46.145,2025-01-01 00:03:52,0,2,1,1,0.0,1.0,0.974928,-0.222521,-5.855
78827,29.0,52553080,34800,2025-01-01 00:06:13.000,2025-01-01 00:05:45,0,2,1,1,0.0,1.0,0.974928,-0.222521,28.000


In [78]:
weather_data = dd.read_csv("../data/raw/weather_yearly/weather_2025_yearly.csv",
                           usecols=['Climate ID', 
                                    'Date/Time (LST)',
                                    'Temp (°C)',
                                    'Dew Point Temp (°C)',
                                    'Rel Hum (%)',
                                    'Wind Spd (km/h)',
                                    'Visibility (km)',
                                    'Stn Press (kPa)'])

In [79]:
weather_data.compute().shape

(8760, 8)

In [80]:
weather_data = weather_data.fillna(0)

In [81]:
weather_data.isna().compute().sum()

Climate ID             0
Date/Time (LST)        0
Temp (°C)              0
Dew Point Temp (°C)    0
Rel Hum (%)            0
Wind Spd (km/h)        0
Visibility (km)        0
Stn Press (kPa)        0
dtype: int64

In [82]:
weather_data['Date/Time (LST)'] = dd.to_datetime(weather_data['Date/Time (LST)'], format='mixed')

In [83]:
weather_data.dtypes

Climate ID                    float64
Date/Time (LST)        datetime64[ns]
Temp (°C)                     float64
Dew Point Temp (°C)           float64
Rel Hum (%)                   float64
Wind Spd (km/h)               float64
Visibility (km)               float64
Stn Press (kPa)               float64
dtype: object

In [84]:
weather_data.head()

,Climate ID,Date/Time (LST),Temp (°C),Dew Point Temp (°C),Rel Hum (%),Wind Spd (km/h),Visibility (km),Stn Press (kPa)
0,6158731,2025-01-01 00:00:00,1.6,1.6,100.0,27.0,16.1,97.93
1,6158731,2025-01-01 01:00:00,1.6,1.6,100.0,29.0,19.3,97.95
2,6158731,2025-01-01 02:00:00,1.4,1.4,100.0,30.0,19.3,97.98
3,6158731,2025-01-01 03:00:00,0.8,0.8,100.0,26.0,9.7,98.03
4,6158731,2025-01-01 04:00:00,1.4,1.4,100.0,23.0,19.3,98.07


In [85]:
# Create nearest_whole_hour column by rounding actual_arrival to the nearest hour
df['nearest_whole_hour'] = df['actual_arrival'].dt.round('h')

In [86]:
weather_data.head()

,Climate ID,Date/Time (LST),Temp (°C),Dew Point Temp (°C),Rel Hum (%),Wind Spd (km/h),Visibility (km),Stn Press (kPa)
0,6158731,2025-01-01 00:00:00,1.6,1.6,100.0,27.0,16.1,97.93
1,6158731,2025-01-01 01:00:00,1.6,1.6,100.0,29.0,19.3,97.95
2,6158731,2025-01-01 02:00:00,1.4,1.4,100.0,30.0,19.3,97.98
3,6158731,2025-01-01 03:00:00,0.8,0.8,100.0,26.0,9.7,98.03
4,6158731,2025-01-01 04:00:00,1.4,1.4,100.0,23.0,19.3,98.07


In [87]:
# Rename weather column to match our nearest_whole_hour for merging
weather_data = weather_data.rename(columns={'Date/Time (LST)': 'nearest_whole_hour'})

In [88]:
weather_data.head()

,Climate ID,nearest_whole_hour,Temp (°C),Dew Point Temp (°C),Rel Hum (%),Wind Spd (km/h),Visibility (km),Stn Press (kPa)
0,6158731,2025-01-01 00:00:00,1.6,1.6,100.0,27.0,16.1,97.93
1,6158731,2025-01-01 01:00:00,1.6,1.6,100.0,29.0,19.3,97.95
2,6158731,2025-01-01 02:00:00,1.4,1.4,100.0,30.0,19.3,97.98
3,6158731,2025-01-01 03:00:00,0.8,0.8,100.0,26.0,9.7,98.03
4,6158731,2025-01-01 04:00:00,1.4,1.4,100.0,23.0,19.3,98.07


In [89]:
# Merge df with weather_data on nearest_whole_hour
print("Starting merge operation...")
start_time = time.time()

# Perform the merge
df_with_weather = df.merge(
    weather_data,
    on='nearest_whole_hour',
    how='left'
)

merge_time = time.time() - start_time
print(f"Merge operation set up. Time elapsed: {merge_time:.2f} seconds")
print(f"Total partitions in merged dataframe: {df_with_weather.npartitions}")

Starting merge operation...
Merge operation set up. Time elapsed: 0.02 seconds
Total partitions in merged dataframe: 142


In [90]:
# Preview the merged data structure (lazy operation)
df_with_weather.head()

,route,trip_id,stopid,thrusteet,actual_arrival,planned_arrival,hour,day_of_week,month,day_of_year,...,day_of_week_cos,delay_seconds,nearest_whole_hour,Climate ID,Temp (°C),Dew Point Temp (°C),Rel Hum (%),Wind Spd (km/h),Visibility (km),Stn Press (kPa)
0,29.0,172354080,3880_0,Wilson Station,2025-01-01 00:01:20.000,2025-01-01 00:02:00,0,2,1,1,...,-0.222521,-40.000,2025-01-01,6158731,1.6,1.6,100.0,27.0,16.1,97.93
1,29.0,52553080,2728_1,Princes' Gates Loop,2025-01-01 00:01:34.029,2025-01-01 00:03:00,0,2,1,1,...,-0.222521,-85.971,2025-01-01,6158731,1.6,1.6,100.0,27.0,16.1,97.93
2,39.0,76911080,3784_1,Finch Station,2025-01-01 00:02:41.000,2025-01-01 00:05:00,0,2,1,1,...,-0.222521,-139.000,2025-01-01,6158731,1.6,1.6,100.0,27.0,16.1,97.93
3,29.0,52553080,3993,Princes' Blvd,2025-01-01 00:03:46.145,2025-01-01 00:03:52,0,2,1,1,...,-0.222521,-5.855,2025-01-01,6158731,1.6,1.6,100.0,27.0,16.1,97.93
4,29.0,52553080,34800,Princes' Blvd,2025-01-01 00:06:13.000,2025-01-01 00:05:45,0,2,1,1,...,-0.222521,28.000,2025-01-01,6158731,1.6,1.6,100.0,27.0,16.1,97.93


In [91]:
# Persist the merged dataframe in distributed memory for faster operations
print("Persisting merged dataframe in distributed memory...")
print(f"Processing {df_with_weather.npartitions} partitions in parallel...")
start_time = time.time()

df_with_weather = df_with_weather.persist()

# Wait for computation to complete and show progress
progress(df_with_weather)

persist_time = time.time() - start_time
print(f"\nPersist operation completed in {persist_time:.2f} seconds ({persist_time/60:.2f} minutes)")

Persisting merged dataframe in distributed memory...
Processing 142 partitions in parallel...

Persist operation completed in 0.22 seconds (0.00 minutes)


In [92]:
# Display final columns
print("Final dataframe columns:")
print(df_with_weather.columns.tolist())

Final dataframe columns:
['route', 'trip_id', 'stopid', 'thrusteet', 'actual_arrival', 'planned_arrival', 'hour', 'day_of_week', 'month', 'day_of_year', 'hour_sin', 'hour_cos', 'day_of_week_sin', 'day_of_week_cos', 'delay_seconds', 'nearest_whole_hour', 'Climate ID', 'Temp (°C)', 'Dew Point Temp (°C)', 'Rel Hum (%)', 'Wind Spd (km/h)', 'Visibility (km)', 'Stn Press (kPa)']


In [93]:
# Drop datetime columns before saving to reduce file size and memory usage
print("Dropping actual_arrival and planned_arrival columns...")
df_to_save = df_with_weather.drop(columns=['actual_arrival', 'planned_arrival', 'nearest_whole_hour'])
# Save the merged dataframe as parquet with parallel processing
output_path_29_and_39 = "./tmp/unfv_with_weather_route_29_39.parquet"
print(f"Saving merged dataframe to {output_path_29_and_39}...")
print(f"Using parallel processing with {df_to_save.npartitions} partitions...")
save_start = time.time()

# Save to parquet format (Dask will write in parallel)
df_to_save.to_parquet(
    output_path_29_and_39,
    engine='pyarrow',
    compression='snappy',  # Good balance between compression and speed
    write_index=False
)

save_time = time.time() - save_start
print("\n✓ Successfully saved to parquet!")
print(f"Save time: {save_time:.2f} seconds ({save_time/60:.2f} minutes)")
print(f"Output location: {output_path_29_and_39 }")

Dropping actual_arrival and planned_arrival columns...
Saving merged dataframe to ./tmp/unfv_with_weather_route_29_39.parquet...
Using parallel processing with 142 partitions...

✓ Successfully saved to parquet!
Save time: 65.16 seconds (1.09 minutes)
Output location: ./tmp/unfv_with_weather_route_29_39.parquet


In [94]:
# Verify the saved parquet file
print("Verifying saved parquet file...")
verify_start = time.time()
df_verify = dd.read_parquet(output_path_29_and_39)
verify_rows = df_verify.shape[0].compute()
verify_cols = len(df_verify.columns)
verify_time = time.time() - verify_start
print(f"Verified rows in parquet file: {verify_rows:,}")
print(f"Number of columns: {verify_cols}")
print(f"Columns: {df_verify.columns.tolist()}")
print(f"Verification time: {verify_time:.2f} seconds")
print("\n✓ All data successfully saved and verified!")
print("\nPreview of first few rows:")
print(df_verify.head())

Verifying saved parquet file...
Verified rows in parquet file: 11,287,394
Number of columns: 20
Columns: ['route', 'trip_id', 'stopid', 'thrusteet', 'hour', 'day_of_week', 'month', 'day_of_year', 'hour_sin', 'hour_cos', 'day_of_week_sin', 'day_of_week_cos', 'delay_seconds', 'Climate ID', 'Temp (°C)', 'Dew Point Temp (°C)', 'Rel Hum (%)', 'Wind Spd (km/h)', 'Visibility (km)', 'Stn Press (kPa)']
Verification time: 0.11 seconds

✓ All data successfully saved and verified!

Preview of first few rows:
   route    trip_id  stopid            thrusteet  hour  day_of_week  month  \
0   29.0  172354080  3880_0       Wilson Station     0            2      1   
1   29.0   52553080  2728_1  Princes' Gates Loop     0            2      1   
2   39.0   76911080  3784_1        Finch Station     0            2      1   
3   29.0   52553080    3993        Princes' Blvd     0            2      1   
4   29.0   52553080   34800        Princes' Blvd     0            2      1   

   day_of_year  hour_sin  hou

In [95]:
# Check final shape
print("Computing final dataframe shape...")
shape_start = time.time()
final_shape = df_with_weather.shape[0].compute()
shape_time = time.time() - shape_start
print(f"Total rows: {final_shape:,}")
print(f"Time taken: {shape_time:.2f} seconds")

Computing final dataframe shape...
Total rows: 11,287,394
Time taken: 0.19 seconds
